In [0]:
import pandas as pd
import polars as pl
import numpy as np
import datetime
import pytz
import os
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
import calendar
from datetime import date
warnings.filterwarnings('ignore')
spark.conf.set("spark.sql.adaptive.enabled", "false") 
ny_tz = pytz.timezone("America/New_York")

In [0]:
ipcap_processed_table = 'opsanalytics_adb_workspace01.capacity_modeling_staging.ipcap_processed_gold'
output_table = 'opsanalytics_adb_workspace01.capacity_modeling_staging.or_ip_simulated_minutes_gold'
or_case_table = 'opsanalytics_adb_workspace01.capacity_modeling.or_quality_dashboard_case_details'
holidays_table = 'opsanalytics_adb_workspace01.capacity_modeling.mshs_holidays'

In [0]:
# 2. Load the Holidays Table natively 
holidays_df = spark.table(holidays_table)
holidays_df = holidays_df.pandas_api()
holidays_df['holiday_date'] = holidays_df['holiday_date'].astype('datetime64[ns]')

In [0]:
def adjust_surgery_dates(df, holiday_df):
    """
    Shifts NEW_SURGERY_DATE forward by 1 day at a time (up to 5 days max) 
    until the date no longer lands on a weekend or a holiday.
    """
    # 1. Convert holiday list to a set of datetime.date objects for lightning-fast lookup
    holiday_set = holidays_df['holiday_date'].to_list()
    
    # 2. Initialize tracking variables
    max_shifts = 5
    shift_count = 0
    
    # Create copies of our target columns to modify safely
    new_dates = df['NEW_SURGERY_DATE'].copy()
    
    while shift_count < max_shifts:
        # Create masks for conditions
        is_weekend = new_dates.dt.dayofweek.isin([5, 6])
        is_holiday = new_dates.dt.date.isin(holiday_set)
        
        # A row needs a shift if it hits either condition
        needs_shift = is_weekend | is_holiday
        
        # If no records land on a weekend or holiday anymore, we are done!
        if not needs_shift.any():
            break
            
        # Move ONLY the problematic dates forward by exactly 1 day
        new_dates.loc[needs_shift] += pd.to_timedelta(1, unit='D')
        shift_count += 1
        
    return new_dates

def join_scenario_and_or_data(ip_sim_data,or_case_table,exclude_holidays_weekends):

    query = f""" 
    with dedup as (
      select distinct ENCOUNTER_NO, MSMRN, 
            try_cast(ADMIT_DT_SRC as timestamp) as ADMIT_DT_SRC,
            try_cast(DSCH_DT_SRC as timestamp) as DSCH_DT_SRC,
            try_cast(NEW_ADMIT_DT_SRC as timestamp) as NEW_ADMIT_DT_SRC,
            try_cast(NEW_DSCH_DT_SRC as timestamp) as NEW_DSCH_DT_SRC,
            SIMULATION, SCENARIO, PROJECTION_YEAR
      from {ip_sim_data} 
      where MSMRN LIKE '%\_%'
    )
    select 
        a.HOSPITAL, 
        a.PATIENT_IN_ROOM_DTTM, 
        a.PATIENT_OUT_ROOM_DTTM, 
        a.SURGERY_DATE, 
        a.SURGEON_SPECIALTY, 
        a.OR_ID, 
        a.OR_LOCATION, 
        a.CASE_STATUS, 
        a.OR_CASE_ID, 
        b.ENCOUNTER_NO, 
        b.MSMRN, 
        element_at(split(b.MSMRN, '_'), 1) as PARENT_MRN,  
        b.ADMIT_DT_SRC, 
        b.DSCH_DT_SRC, 
        b.NEW_ADMIT_DT_SRC, 
        b.NEW_DSCH_DT_SRC,
        b.SCENARIO,
        b.SIMULATION 
    from {or_case_table} a 
    join dedup b 
      on a.PAT_MRN_ID = element_at(split(b.MSMRN, '_'), 1) 
      and a.SURGERY_DATE between b.admit_dt_src and b.dsch_dt_src 
    where a.patient_in_room_dttm is not null 
    """

    df = spark.sql(query)
    df = df.toPandas()
    df['actual_los'] = df['DSCH_DT_SRC'] - df['ADMIT_DT_SRC']
    df['actual_los'] = df['actual_los'].dt.days
    df['new_los'] = df['NEW_DSCH_DT_SRC'] - df['NEW_ADMIT_DT_SRC']
    df['new_los'] = df['new_los'].dt.days
    df['surgery_offset'] = df['SURGERY_DATE'] - df['ADMIT_DT_SRC']
    df['surgery_offset'] = df['surgery_offset'].dt.days


    df['NEW_SURGERY_DATE'] = df['NEW_ADMIT_DT_SRC'] + pd.to_timedelta(df['surgery_offset'], unit='D')
    df['NEW_SURGERY_YEAR'] = df['NEW_SURGERY_DATE'].dt.year
    date_shift = df['NEW_SURGERY_DATE'] - df['SURGERY_DATE']
    df['NEW_SURGERY_DATE'] = adjust_surgery_dates(df, holidays_df)
    df['NEW_PATIENT_IN_ROOM_DTTM'] = df['PATIENT_IN_ROOM_DTTM'] + date_shift
    df['NEW_PATIENT_OUT_ROOM_DTTM'] = df['PATIENT_OUT_ROOM_DTTM'] + date_shift
    df['case_minutes'] = df['PATIENT_OUT_ROOM_DTTM'] - df['PATIENT_IN_ROOM_DTTM']
    df['case_minutes'] = df['case_minutes'].dt.total_seconds()/60
    print(df.shape)
    if exclude_holidays_weekends:
      # Exclude Weekends
      df = df.loc[~df['NEW_SURGERY_DATE'].dt.dayofweek.isin([5,6])]
      # Exclude Holidays
      df = df.loc[~df['NEW_SURGERY_DATE'].isin(holidays_df['holiday_date'].to_list())]
    print(df.shape)


    return df

In [0]:
def get_case_minutes_simulation(simulation_data):
    # 1. Filter out MSBI early to free up memory before aggregating
    filtered_data = simulation_data[simulation_data['HOSPITAL'] != 'MSBI']
    
    # 2. Group by and aggregate both metrics in a single pass
    total_case_minutes = (
        filtered_data.groupby(["HOSPITAL", "SCENARIO", "SIMULATION", "NEW_SURGERY_YEAR"])
        .agg(
            case_minutes=('case_minutes', 'sum')        )
        .reset_index()
    )
    total_case_minutes.rename(columns={'NEW_SURGERY_YEAR': 'year'}, inplace=True)

    total_case_minutes['CREATED_DATETIME'] = datetime.datetime.now(ny_tz)
    # total_case_minutes['surgery_days'] = 365 + calendar.isleap(total_case_minutes['year'])

    # 3. Calculate the average minutes directly
    # total_case_minutes['avg_minutes'] = total_case_minutes['case_minutes'] / total_case_minutes['NEW_SURGERY_DATE']
    
    return total_case_minutes

In [0]:
simualtion_or_cases = join_scenario_and_or_data(ipcap_processed_table,
                                                or_case_table,
                                                exclude_holidays_weekends=True)

In [0]:
# display(simualtion_or_cases)

In [0]:
simulation_case_minutes = get_case_minutes_simulation(simualtion_or_cases)

In [0]:
spark.createDataFrame(simulation_case_minutes).write \
.mode("append") \
.option("mergeSchema", "true") \
.format("delta") \
.saveAsTable(output_table)